# Decorator Pattern: Logging and Timing

## Objective

Learn how to add reusable functionality to existing methods without changing their original implementation, and understand how this supports the **Open/Closed Principle**.

In this notebook, we use a random employee CSV dataset and clean it with pandas. The dataset is only a practical example; the main topic is decorators.

## 1. What is a decorator?

A **decorator** is a function that wraps another function or method to add behavior before or after it runs.

For example, a data-cleaning method can clean a DataFrame. Decorators can add logging and execution-time measurement around that method without putting those extra tasks into the cleaning logic.

- `@log_action`: logs when a method starts and finishes.
- `@log_time`: measures how long a method takes.
- `@wraps`: preserves the wrapped function's metadata, such as its name and docstring.

Conceptually:

**Original method → decorator wrapper → original behavior + extra functionality**

## 2. The Open/Closed Principle

The **Open/Closed Principle** says that software entities should be:

- **Open for extension:** new behavior can be added.
- **Closed for modification:** existing, stable code should not need to change to add that behavior.

Here, `DataCleaner.clean()` focuses on cleaning data. Logging and timing are added with decorators, so the cleaning logic does not need to be rewritten to include those features.

Decorators are one way to support this principle; they do not guarantee that every design is automatically closed to modification.

## 3. Create the decorators

### `@log_action`

This decorator prints a start message before calling the original function and a finish message afterward. The `finally` block ensures the finish message is printed even if the wrapped function raises an exception.

### `@log_time`

This decorator records a start time, runs the original function, then calculates elapsed time with `time.perf_counter()`.

### `@wraps(func)`

Without `@wraps(func)`, the decorated function can appear to have the name `wrapper` and may lose its original docstring. `@wraps` preserves useful metadata from the original function.

In [19]:
import time
from functools import wraps

def log_action(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"[START] {func.__name__}")
        try:
            return func(*args, **kwargs)
        finally:
            print(f"[FINISH] {func.__name__}")
    return wrapper


def log_time(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[TIME] {func.__name__} took {elapsed:.4f} seconds")
        return result
    return wrapper

## 4. Generate and load the random dataset

The CSV contains intentionally imperfect employee data, such as missing values, inconsistent capitalization, invalid ages or salaries, and duplicate rows. These issues give us something realistic to clean.

Place `random_employee_data.csv` in the same folder as this notebook before running the next cell.

In [20]:
import pandas as pd

df = pd.read_csv("random_employee_data.csv")

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (32, 6)


,employee_id,name,age,department,salary_usd,city
0,1,BOB,19.0,HR,1768.0,Siem Reap
1,2,george,20.0,Finance,311.0,Siem Reap
2,3,george,32.0,finance,NaN,Battambang
3,4,Charlie,NaN,HR,NaN,Phnom Penh
4,5,Hannah,52.0,Sales,505.0,Phnom Penh


## 5. Build a data cleaner with decorators

The `clean()` method contains the data-cleaning rules. The decorators add logging and timing externally.

Decorator order matters: with `@log_action` above `@log_time`, the logging wrapper surrounds the timing wrapper. The start message appears first, the timing message appears after the method runs, and the finish message appears last.

In [21]:
class DataCleaner:
    @log_action
    @log_time
    def clean(self, df):
        """Return a cleaned copy of the input DataFrame."""
        df = df.copy()

        print(f"Rows before cleaning: {len(df)}")

        # Remove duplicate rows
        df = df.drop_duplicates()

        # Standardize text columns
        for col in ["name", "department", "city"]:
            df[col] = df[col].astype("string").str.strip()

        df["name"] = df["name"].str.title()
        df["department"] = df["department"].str.title()
        df["city"] = df["city"].str.title()

        # Treat impossible values as missing
        df.loc[~df["age"].between(18, 100), "age"] = pd.NA
        df.loc[df["salary_usd"] < 0, "salary_usd"] = pd.NA
        df.loc[df["salary_usd"] > 100_000, "salary_usd"] = pd.NA

        # Fill missing text fields
        for col in ["name", "department", "city"]:
            df[col] = df[col].fillna("Unknown")

        # Fill missing numeric values with the median
        df["age"] = df["age"].fillna(df["age"].median())
        df["salary_usd"] = df["salary_usd"].fillna(
            df["salary_usd"].median()
        )

        # Keep output types practical
        df["age"] = df["age"].round().astype(int)
        df["salary_usd"] = df["salary_usd"].round(2)

        print(f"Rows after cleaning: {len(df)}")
        return df

## 6. Run the cleaning method

The original DataFrame is passed to `clean()`. The method creates a copy, so the raw DataFrame remains unchanged.

In [22]:
cleaner = DataCleaner()
cleaned_df = cleaner.clean(df)

display(cleaned_df.head())

[START] clean
Rows before cleaning: 32
Rows after cleaning: 30
[TIME] clean took 0.0139 seconds
[FINISH] clean


,employee_id,name,age,department,salary_usd,city
0,1,Bob,19,Hr,1768.0,Siem Reap
1,2,George,20,Finance,311.0,Siem Reap
2,3,George,32,Finance,1196.0,Battambang
3,4,Charlie,42,Hr,1196.0,Phnom Penh
4,5,Hannah,52,Sales,505.0,Phnom Penh


## 7. Compare before and after cleaning

We can check the dataset dimensions and missing values to see the effect of the cleaning rules.

In [23]:
print("Original shape:", df.shape)
print("Cleaned shape:", cleaned_df.shape)

print("\nMissing values before cleaning:")
display(df.isna().sum().to_frame("missing_before"))

print("\nMissing values after cleaning:")
display(cleaned_df.isna().sum().to_frame("missing_after"))

Original shape: (32, 6)
Cleaned shape: (30, 6)

Missing values before cleaning:


,missing_before
employee_id,0
name,3
age,1
department,5
salary_usd,3
city,8



Missing values after cleaning:


,missing_after
employee_id,0
name,0
age,0
department,0
salary_usd,0
city,0


## 8. Why use `@wraps`?

`@wraps(func)` helps preserve the original function's metadata after decoration. This is useful for debugging, documentation, and inspection.

For example, we can inspect the decorated method's name and docstring.

In [24]:
print("Method name:", cleaner.clean.__name__)
print("Method docstring:", cleaner.clean.__doc__)

Method name: clean
Method docstring: Return a cleaned copy of the input DataFrame.


## 9. Key takeaways

1. **Decorators wrap functions or methods** to add behavior.
2. **`@log_action`** adds start and finish messages.
3. **`@log_time`** measures elapsed execution time using `time.perf_counter()`.
4. **`@wraps(func)`** preserves metadata such as the original name and docstring.
5. **Reusability:** the same decorators can be applied to methods in different classes.
6. **Open/Closed Principle:** we can extend methods with logging and timing without inserting that code into the original data-cleaning logic.

The CSV cleaning is the example. The main lesson is how decorators add reusable functionality around existing code.